# Public OrPen Xmon — Route B Electrostatic

This public workflow prepares a Palace manual handoff. It does not
submit or run Palace.

In [ ]:
from pathlib import Path

import gdsfactory as gf
from scgsim.palace import ElectrostaticSim, inspect_run_trustworthiness, resolve_palace_result
from scgsim.sgb import build_component_stack
from scgsim.visualization import inspect_palace_geometry

import orpen_sc_pdk
from orpen_sc_pdk import (
    LAYER_STACK,
    get_interface_preset_records,
    get_material_records,
    validate_interface_preset_records,
)
from orpen_sc_pdk.helpers.assembly import place_flip_chip_ground_short_bumps
from orpen_sc_pdk.tech import OUTER_VACUUM_THICKNESS_UM

# Choose prepare_handoff to prepare a manual run or analyze_handoff to inspect returned results.
WORKFLOW_ACTION = "prepare_handoff"
# Use a unique ID for each new prepared run; SCGSim refuses non-empty output directories.
RUN_ID = "kosen2024_xmon_route_b_es_l309p5_w24p65_g20_litcentral_fem2_20260903_04"
RUN_ROOT = Path.cwd() / ".artifacts" / RUN_ID  # Root for this run's artifacts.
# Exact ID returned by Prepare Handoff; paste it here before analyzing a returned run.
EXPECTED_HANDOFF_ID = ""
if WORKFLOW_ACTION not in {"prepare_handoff", "analyze_handoff"}:
    raise ValueError("WORKFLOW_ACTION must be 'prepare_handoff' or 'analyze_handoff'.")

## Build Component Coupon

In [ ]:
if WORKFLOW_ACTION == "prepare_handoff":
    COMPONENT_NAME = "kosen2024_flip_chip_xmon_qubit"
    COMPONENT_PARAMETERS = {
        "bump_ring_count_per_side": 0,
        "qubit_pad_length": 309.5,
        "qubit_pad_width": 24.65,
        "qubit_gap": 20.0,
    }
    COUPON_PADDING_UM = 75.0
    GROUND_SHORT_CLEARANCE_UM = 30.0
    INDIUM_PLACEMENT_MODE = "corner_anchors"

    orpen_sc_pdk.activate()
    gf.clear_cache()
    component = gf.get_component(COMPONENT_NAME, **COMPONENT_PARAMETERS)
    coupon = place_flip_chip_ground_short_bumps(
        component,
        coupon_padding_um=COUPON_PADDING_UM,
        clearance_um=GROUND_SHORT_CLEARANCE_UM,
        placement_mode=INDIUM_PLACEMENT_MODE,
    )
    component = coupon.component
    STACK_COUPON_PADDING_UM = coupon.stack_coupon_padding_um
    coupon.plot()

## Configure EPR / Problem

The MA/MS/SA inputs use OrPen’s synthetic literature-central presets;
they are modeling inputs, not measured process data or material
properties. Each preset records the ordinary-median epsilon_r and
thickness, the log-space-median loss tangent, the excluded
assumed-typical and censored-upper-bound rows, and the 2 nm thickness
modeling convention in its PDK provenance.

In [ ]:
if WORKFLOW_ACTION == "prepare_handoff":
    ROUTE = "B"
    # Add symmetric X/Y/Z padding to the automatic vacuum envelope; only Z is expanded here.
    VACUUM_PADDING_UM = (0.0, 0.0, float(OUTER_VACUUM_THICKNESS_UM))
    INDIUM_GROUND_FILL = {
        "fill": False,
        "fill_pitch_um": 80.0,
        "fill_clearance_um": 30.0,
    }
    # The Xmon pad is the sole driven terminal. Couplers, both ground planes,
    # and the corner-authored bumps form one RF-grounded conductor group.
    TERMINALS = {
        "xmon_pad": "xmon_pad",  # Report name -> exact structured conductor net.
    }
    SAVE_FIELDS = 0
    UNASSIGNED_CONDUCTOR_POLICY = "ground"
    # Leave exterior solution boundaries at Palace's natural zero-charge condition.
    EXTERIOR_BOUNDARY_POLICY = "none"

    stack = build_component_stack(
        component=component,
        layer_stack=LAYER_STACK,
        material_records=get_material_records(),
        coupon_padding_um=STACK_COUPON_PADDING_UM,
    )
    sim = ElectrostaticSim()
    sim.set_geometry(component)
    sim.set_stack(stack)
    sim.set_output_dir(RUN_ROOT)
    sim.set_vacuum_region(padding=VACUUM_PADDING_UM)
    sim.set_indium_ground_bumps(**INDIUM_GROUND_FILL)
    # Select complete validated OrPen records immediately before Surface-EPR setup so
    # their public source/description provenance remains visible beside the numeric specs.
    records = validate_interface_preset_records(get_interface_preset_records())
    EPR_PRESET_NAMES = {
        "MA": "LiteratureCentral_MA",
        "MS": "LiteratureCentral_MS",
        "SA": "LiteratureCentral_SA",
    }
    missing_preset_names = sorted(set(EPR_PRESET_NAMES.values()) - records.keys())
    if missing_preset_names:
        raise KeyError(f"Missing required OrPen interface presets: {missing_preset_names}")
    selected_interface_presets = {
        interface_type: records[preset_name]
        for interface_type, preset_name in EPR_PRESET_NAMES.items()
    }
    for interface_type, preset in selected_interface_presets.items():
        if preset["interface_type"] != interface_type:
            raise ValueError(
                f"OrPen preset {EPR_PRESET_NAMES[interface_type]!r} has interface_type "
                f"{preset['interface_type']!r}; expected {interface_type!r}."
            )
    EPR_SPECS = {
        interface_type: {
            "thickness": preset["thickness"],
            "permittivity": preset["permittivity"],
            "loss_tangent": preset["loss_tangent"],
        }
        for interface_type, preset in selected_interface_presets.items()
    }
    sim.set_surface_epr(representation=ROUTE, specs=EPR_SPECS)
    for terminal_name, net_id in TERMINALS.items():
        sim.add_terminal(terminal_name, net_id=net_id)
    sim.set_electrostatic(
        save_fields=SAVE_FIELDS,
        unassigned_conductor_policy=UNASSIGNED_CONDUCTOR_POLICY,
        exterior_boundary_policy=EXTERIOR_BOUNDARY_POLICY,
    )

## Build Mesh

In [ ]:
if WORKFLOW_ACTION == "prepare_handoff":
    REFINED_MESH_SIZE_UM = 5.0
    MAX_MESH_SIZE_UM = 300.0

    sim.set_mesh(
        refined_mesh_size=REFINED_MESH_SIZE_UM,
        max_mesh_size=MAX_MESH_SIZE_UM,
    )
    MESH_PATH = sim.mesh()

## Generate Config

In [ ]:
if WORKFLOW_ACTION == "prepare_handoff":
    FEM_ORDER = 2
    LINEAR_TOLERANCE = 1e-6
    MAX_ITERATIONS = 2000
    SOLVER_TYPE = "Default"
    PRECONDITIONER = "Default"
    DEVICE = "CPU"
    AMR_MAX_PASSES = 20
    AMR_NONCONFORMAL = False
    AMR_TOLERANCE = 2e-2
    AMR_UPDATE_FRACTION = 0.15
    SAVE_ADAPT_ITERATIONS = True
    ESTIMATOR_MG = False
    OUTPUT_PARAVIEW = False
    OUTPUT_GRID_FUNCTION = False

    sim.set_numerical(
        order=FEM_ORDER,
        tolerance=LINEAR_TOLERANCE,
        max_iterations=MAX_ITERATIONS,
        solver_type=SOLVER_TYPE,
        preconditioner=PRECONDITIONER,
        device=DEVICE,
        amr_max_passes=AMR_MAX_PASSES,
        amr_nonconformal=AMR_NONCONFORMAL,
        amr_tolerance=AMR_TOLERANCE,
        amr_update_fraction=AMR_UPDATE_FRACTION,
        save_adapt_iterations=SAVE_ADAPT_ITERATIONS,
        estimator_mg=ESTIMATOR_MG,
        output_paraview=OUTPUT_PARAVIEW,
        output_grid_function=OUTPUT_GRID_FUNCTION,
    )
    CONFIG_PATH = sim.write_config()

## Prepare Handoff

In [ ]:
if WORKFLOW_ACTION == "prepare_handoff":
    MACHINE_PROFILE = "direct-local"
    PALACE_EXECUTABLE = "palace"
    SETUP_COMMANDS = ("module load palace",)
    RESOURCES = {
        "processes": 32,
        "threads": 2,
        "command_style": "wrapper",
    }

    HANDOFF = sim.prepare_handoff(
        profile=MACHINE_PROFILE,
        executable=PALACE_EXECUTABLE,
        resources=RESOURCES,
        setup_commands=SETUP_COMMANDS,
    )

## Analyze Returned Run

In [ ]:
# Returned handoff root whose receipt and handoff identity SCGSim must verify.
RETURNED_RUN_DIR = RUN_ROOT
if WORKFLOW_ACTION == "analyze_handoff":
    PREVIEW_MODE = "boundaries"  # materials | boundaries | surface_epr | mesh
    report = inspect_run_trustworthiness(RETURNED_RUN_DIR)
    if report.completeness == "complete":
        report = resolve_palace_result(RETURNED_RUN_DIR, expected_handoff_id=EXPECTED_HANDOFF_ID)
    report.show_all_results()
    preview = inspect_palace_geometry(RETURNED_RUN_DIR)
    preview.explore(PREVIEW_MODE)